In [1]:
# Import libraries and set project root path
import pandas as pd
import numpy as np
import os

BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Base path:", BASE)

Base path: /Users/alexia/Documents/CASA/Dissertation


In [2]:
# Reload all datasets so this notebook runs independently without 01_data_loading.ipynb

# TS001
ts001 = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/TS001_usual_residents_London_LSOA_2021.csv"), skiprows=7)
ts001 = ts001[ts001["Area"].str.contains("lsoa2021:", na=False)].copy()
ts001["lsoa_code"] = ts001["Area"].str.extract(r"lsoa2021:(E\d+)")
ts001["lsoa_name"] = ts001["Area"].str.extract(r"E\d+ : (.+)$")
ts001 = ts001.drop(columns=["Area"]).rename(columns={"2021": "total_residents"})
ts001 = ts001[["lsoa_code", "lsoa_name", "total_residents"]].reset_index(drop=True)

# TS045
ts045 = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv"), skiprows=7)
ts045 = ts045[ts045["Area"].str.contains("lsoa2021:", na=False)].copy()
ts045["lsoa_code"] = ts045["Area"].str.extract(r"lsoa2021:(E\d+)")
ts045["lsoa_name"] = ts045["Area"].str.extract(r"E\d+ : (.+)$")
ts045 = ts045.drop(columns=["Area"]).rename(columns={
    "No cars or vans in household":        "cars_0",
    "1 car or van in household":           "cars_1",
    "2 cars or vans in household":         "cars_2",
    "3 or more cars or vans in household": "cars_3plus"
})
ts045 = ts045[["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]].reset_index(drop=True)

# IMD 2019
xl = pd.ExcelFile(os.path.join(BASE, "03_data/demand/imd2019/IMD2019_LSOA_England.xlsx"))
imd = xl.parse("IMD2019").rename(columns={
    "LSOA code (2011)":                           "lsoa_code_2011",
    "LSOA name (2011)":                           "lsoa_name",
    "Local Authority District code (2019)":       "lad_code",
    "Local Authority District name (2019)":       "lad_name",
    "Index of Multiple Deprivation (IMD) Rank":   "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile": "imd_decile"
})

# OpenStreetEV GLA
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))
osev = osev_raw[["country_co","id","external_l","name","address","postal_cod","state","coordinate","coordina_1","location_c","location_1"]].copy()
osev = osev.rename(columns={
    "country_co": "country_code", "id": "location_id", "external_l": "external_uuid",
    "name": "location_name", "address": "address", "postal_cod": "postcode",
    "state": "borough", "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "location_1": "location_type"
})

# GLA Join, EVSE Status, Device UK
gla_join    = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/gla_location_evse_join.csv"))
evse_status = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/evse_status_gla.csv"))
device      = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/device_all_uk.csv"), low_memory=False)

print("All datasets reloaded successfully.")

All datasets reloaded successfully.


In [3]:
# Create output directory for cleaned data if it does not exist
output_dir = os.path.join(BASE, "05_processed")
os.makedirs(output_dir, exist_ok=True)
print("Output directory:", output_dir)

Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed


## Filter TS001 and TS045 to London LSOAs only

In [4]:
# Filter TS001 and TS045 to London LSOAs only
# London LSOA codes all start with 'E01' and fall within the range E01000001–E01033768
# We use the lsoa_name to filter: London LSOAs contain borough names, not county names
# Simpler approach: keep only codes that appear in both datasets (both downloaded for London)
# Since we downloaded London-only from Nomis, all rows should already be London
# Verify by checking lsoa_name for known London boroughs

print("=== TS001 sample lsoa_name values ===")
print(ts001["lsoa_name"].head(10).tolist())
print()
print("=== TS045 sample lsoa_name values ===")
print(ts045["lsoa_name"].head(10).tolist())
print()
print("TS001 unique row count:", len(ts001))
print("TS045 unique row count:", len(ts045))

=== TS001 sample lsoa_name values ===
['Hartlepool 001A', 'Hartlepool 001B', 'Hartlepool 001C', 'Hartlepool 001D', 'Hartlepool 001F', 'Hartlepool 001G', 'Hartlepool 002A', 'Hartlepool 002B', 'Hartlepool 002C', 'Hartlepool 002D']

=== TS045 sample lsoa_name values ===
['Hartlepool 001A', 'Hartlepool 001B', 'Hartlepool 001C', 'Hartlepool 001D', 'Hartlepool 001F', 'Hartlepool 001G', 'Hartlepool 002A', 'Hartlepool 002B', 'Hartlepool 002C', 'Hartlepool 002D']

TS001 unique row count: 35672
TS045 unique row count: 35672


In [7]:
# Define all 33 London boroughs (32 boroughs + City of London)
london_boroughs = [
    "City of London", "Barking and Dagenham", "Barnet", "Bexley", "Brent",
    "Bromley", "Camden", "Croydon", "Ealing", "Enfield", "Greenwich",
    "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow", "Havering",
    "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
    "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
    "Redbridge", "Richmond upon Thames", "Southwark", "Sutton",
    "Tower Hamlets", "Waltham Forest", "Wandsworth", "Westminster"
]

# Use regex word-boundary match to avoid 'Brent' matching 'Brentwood'
# Pattern: borough name followed by a space and digits (e.g. 'Brent 001A')
import re

def is_london(name, boroughs):
    if not isinstance(name, str):
        return False
    # Match borough name followed by space + digits (LSOA name format)
    return any(re.match(rf"^{re.escape(b)}\s+\d", name) for b in boroughs)

ts001_london = ts001[ts001["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)
ts045_london = ts045[ts045["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)

print("TS001 London rows:", len(ts001_london))
print("TS045 London rows:", len(ts045_london))
print()
print("TS001 sample:")
print(ts001_london.head(3).to_string())
print()
# Verify Brentwood is gone
brentwood_check = ts001_london[ts001_london["lsoa_name"].str.startswith("Brentwood")]
print("Brentwood rows remaining:", len(brentwood_check))

TS001 London rows: 4994
TS045 London rows: 4994

TS001 sample:
   lsoa_code            lsoa_name total_residents
0  E01000001  City of London 001A            1475
1  E01000002  City of London 001B            1384
2  E01000003  City of London 001C            1613

Brentwood rows remaining: 0


## Cleanup TS045: Convert float to int, calculate vehicle statistics columns

In [10]:
# Convert cars_0 from str to int, then convert remaining float cols
ts045_london["cars_0"] = ts045_london["cars_0"].astype(int)

# Calculate total households per LSOA
ts045_london["total_households"] = (
    ts045_london["cars_0"] +
    ts045_london["cars_1"] +
    ts045_london["cars_2"] +
    ts045_london["cars_3plus"]
)

# Calculate total number of cars per LSOA (weighted sum)
ts045_london["total_cars"] = (
    ts045_london["cars_1"] * 1 +
    ts045_london["cars_2"] * 2 +
    ts045_london["cars_3plus"] * 3  # conservative: treat 3+ as 3
)

# Calculate car ownership rate: proportion of households with at least one car
ts045_london["car_ownership_rate"] = (
    (ts045_london["total_households"] - ts045_london["cars_0"]) /
    ts045_london["total_households"]
).round(4)

print("=== TS045 London Cleaned ===")
print("Shape:", ts045_london.shape)
print(ts045_london.head(3).to_string())
print()
print("car_ownership_rate range:", ts045_london["car_ownership_rate"].min(), "–", ts045_london["car_ownership_rate"].max())

=== TS045 London Cleaned ===
Shape: (4994, 9)
   lsoa_code            lsoa_name  cars_0  cars_1  cars_2  cars_3plus  total_households  total_cars  car_ownership_rate
0  E01000001  City of London 001A     555     243      29          10               837         331              0.3369
1  E01000002  City of London 001B     578     208      26          12               824         296              0.2985
2  E01000003  City of London 001C     826     169      15           7              1017         220              0.1878

car_ownership_rate range: 0.1024 – 0.9602


## Merge TS001 and TS045, then save the cleaned-up file

In [11]:
# Merge TS001 and TS045 on lsoa_code to create a single demand-side census table
census_london = ts001_london.merge(
    ts045_london.drop(columns=["lsoa_name"]),  # drop duplicate lsoa_name column
    on="lsoa_code",
    how="inner"
)

print("=== Census London Merged ===")
print("Shape:", census_london.shape)
print("Columns:", census_london.columns.tolist())
print(census_london.head(3).to_string())
print()
print("Rows lost in merge:", len(ts001_london) - len(census_london))

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/census_london_clean.csv")
census_london.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Census London Merged ===
Shape: (4994, 10)
Columns: ['lsoa_code', 'lsoa_name', 'total_residents', 'cars_0', 'cars_1', 'cars_2', 'cars_3plus', 'total_households', 'total_cars', 'car_ownership_rate']
   lsoa_code            lsoa_name total_residents  cars_0  cars_1  cars_2  cars_3plus  total_households  total_cars  car_ownership_rate
0  E01000001  City of London 001A            1475     555     243      29          10               837         331              0.3369
1  E01000002  City of London 001B            1384     578     208      26          12               824         296              0.2985
2  E01000003  City of London 001C            1613     826     169      15           7              1017         220              0.1878

Rows lost in merge: 0

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/census_london_clean.csv


##  Cleaning IMD 2019, filtering London LSOA

In [12]:
# Filter IMD 2019 to London LSOAs only using the same borough name matching logic
# Note: IMD uses 2011 LSOA boundaries, so lsoa_name format is the same
imd_london = imd[imd["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)

print("=== IMD 2019 London ===")
print("Shape:", imd_london.shape)
print(imd_london.head(3).to_string())
print()

# Check for any missing values
print("Missing values:")
print(imd_london.isnull().sum())

=== IMD 2019 London ===
Shape: (4835, 6)
  lsoa_code_2011            lsoa_name   lad_code        lad_name  imd_rank  imd_decile
0      E01000001  City of London 001A  E09000001  City of London     29199           9
1      E01000002  City of London 001B  E09000001  City of London     30379          10
2      E01000003  City of London 001C  E09000001  City of London     14915           5

Missing values:
lsoa_code_2011    0
lsoa_name         0
lad_code          0
lad_name          0
imd_rank          0
imd_decile        0
dtype: int64


## Download the LSOA 2011–2021 lookup table and merge it

In [14]:
# Load and inspect the LSOA 2011 to 2021 lookup table
lookup = pd.read_csv(os.path.join(BASE, "03_data/demand/census2021/lsoa_2011_to_2021_lookup.csv"))
print("Shape:", lookup.shape)
print("Columns:", lookup.columns.tolist())
lookup.head(3)

Shape: (34753, 9)
Columns: ['LSOA11CD', 'LSOA11NM', 'LSOA21CD', 'LSOA21NM', 'LSOA21NMW', 'LAD22CD', 'LAD22NM', 'LAD22NMW', 'ObjectId']


,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,LSOA21NMW,LAD22CD,LAD22NM,LAD22NMW,ObjectId
0,E01000305,Barnet 036B,E01000305,Barnet 036B,NaN,E09000003,Barnet,NaN,1
1,E01000306,Barnet 036C,E01000306,Barnet 036C,NaN,E09000003,Barnet,NaN,2
2,E01000307,Barnet 039D,E01000307,Barnet 039D,NaN,E09000003,Barnet,NaN,3


In [15]:
# Keep only the mapping columns needed: 2011 code → 2021 code
lookup_slim = lookup[["LSOA11CD", "LSOA21CD"]].drop_duplicates()

# Merge IMD London with lookup to get 2021 LSOA codes
imd_london = imd_london.merge(
    lookup_slim,
    left_on="lsoa_code_2011",
    right_on="LSOA11CD",
    how="left"
)

# Rename the new 2021 code column
imd_london = imd_london.rename(columns={"LSOA21CD": "lsoa_code_2021"})
imd_london = imd_london.drop(columns=["LSOA11CD"])

# Check how many rows got a 2021 code successfully
matched = imd_london["lsoa_code_2021"].notna().sum()
unmatched = imd_london["lsoa_code_2021"].isna().sum()
print(f"Matched: {matched} / Unmatched: {unmatched}")
print()

# Reorder columns
imd_london = imd_london[["lsoa_code_2011", "lsoa_code_2021", "lsoa_name",
                          "lad_code", "lad_name", "imd_rank", "imd_decile"]]

print("=== IMD 2019 London with 2021 codes ===")
print("Shape:", imd_london.shape)
print(imd_london.head(3).to_string())

Matched: 4835 / Unmatched: 0

=== IMD 2019 London with 2021 codes ===
Shape: (4835, 7)
  lsoa_code_2011 lsoa_code_2021            lsoa_name   lad_code        lad_name  imd_rank  imd_decile
0      E01000001      E01000001  City of London 001A  E09000001  City of London     29199           9
1      E01000002      E01000002  City of London 001B  E09000001  City of London     30379          10
2      E01000003      E01000003  City of London 001C  E09000001  City of London     14915           5


## Check LSOA count difference between IMD and Census

In [16]:
# Census has 4994 LSOAs, IMD has 4835 — check which LSOAs are missing from IMD
census_codes = set(census_london["lsoa_code"])
imd_codes = set(imd_london["lsoa_code_2021"])

in_census_not_imd = census_codes - imd_codes
in_imd_not_census = imd_codes - census_codes

print(f"LSOAs in Census but not in IMD: {len(in_census_not_imd)}")
print(f"LSOAs in IMD but not in Census: {len(in_imd_not_census)}")
print()
print("Sample codes in Census but missing from IMD:")
print(list(in_census_not_imd)[:10])

LSOAs in Census but not in IMD: 181
LSOAs in IMD but not in Census: 0

Sample codes in Census but missing from IMD:
['E01034476', 'E01034393', 'E01034590', 'E01034178', 'E01034044', 'E01033919', 'E01033929', 'E01034045', 'E01035701', 'E01034213']


## Fill missing IMD values for new 2021 LSOAs using parent 2011 LSOA

In [17]:
# The 181 missing LSOAs are new 2021 LSOAs split from 2011 LSOAs
# Strategy: assign them the IMD value of their parent 2011 LSOA

# Get the full lookup for London: 2021 code → 2011 code
lookup_london = lookup[lookup["LSOA21CD"].isin(census_codes)][["LSOA21CD", "LSOA11CD"]].drop_duplicates()

# Build IMD reference using 2011 codes
imd_ref = imd_london[["lsoa_code_2011", "imd_rank", "imd_decile"]].drop_duplicates()

# Merge census LSOAs with lookup to get their 2011 parent code
census_with_2011 = census_london[["lsoa_code"]].merge(
    lookup_london,
    left_on="lsoa_code",
    right_on="LSOA21CD",
    how="left"
)

# Then merge with IMD using the 2011 parent code
census_with_imd = census_with_2011.merge(
    imd_ref,
    left_on="LSOA11CD",
    right_on="lsoa_code_2011",
    how="left"
)

# Check how many still missing after parent-code fill
still_missing = census_with_imd["imd_rank"].isna().sum()
print(f"LSOAs still missing IMD after parent-code fill: {still_missing}")
print()
print("Sample:")
print(census_with_imd.head(5).to_string())

LSOAs still missing IMD after parent-code fill: 181

Sample:
   lsoa_code   LSOA21CD   LSOA11CD lsoa_code_2011  imd_rank  imd_decile
0  E01000001  E01000001  E01000001      E01000001   29199.0         9.0
1  E01000002  E01000002  E01000002      E01000002   30379.0        10.0
2  E01000003  E01000003  E01000003      E01000003   14915.0         5.0
3  E01000005  E01000005  E01000005      E01000005    8678.0         3.0
4  E01032739  E01032739  E01032739      E01032739   20391.0         7.0


In [19]:
# Examine the 181 missing LSOA codes to find a pattern
missing_list = sorted(list(in_census_not_imd))
print("First 20 missing codes:")
print(missing_list[:20])
print()

# Check their code range
codes_int = [int(c[3:]) for c in missing_list]
print(f"Code number range: {min(codes_int)} – {max(codes_int)}")
print()

# Compare with IMD codes range
imd_codes_int = [int(c[3:]) for c in imd_london["lsoa_code_2011"]]
print(f"IMD code number range: {min(imd_codes_int)} – {max(imd_codes_int)}")

First 20 missing codes:
['E01033785', 'E01033786', 'E01033787', 'E01033789', 'E01033792', 'E01033862', 'E01033865', 'E01033866', 'E01033868', 'E01033870', 'E01033871', 'E01033874', 'E01033875', 'E01033876', 'E01033880', 'E01033883', 'E01033884', 'E01033911', 'E01033913', 'E01033915']

Code number range: 33785 – 35720

IMD code number range: 1 – 33746


## Use borough-level median IMD to fill missing LSOAs

In [20]:
# These 181 LSOAs are new 2021 boundaries with no IMD 2019 equivalent
# Strategy: fill with borough-level median IMD rank and decile
# This is standard practice in UK spatial analysis literature

# Build borough median from existing IMD London data
borough_median = imd_london.groupby("lad_name")[["imd_rank", "imd_decile"]].median().round(0).astype(int).reset_index()
borough_median.columns = ["lad_name", "imd_rank_median", "imd_decile_median"]

print("Borough median IMD (sample):")
print(borough_median.head(5).to_string())

Borough median IMD (sample):
               lad_name  imd_rank_median  imd_decile_median
0  Barking and Dagenham             6376                  2
1                Barnet            20414                  7
2                Bexley            21124                  7
3                 Brent            12061                  4
4               Bromley            24247                  8


## Build complete IMD table for all 4,994 Census LSOAs

In [21]:
# Build a complete IMD table covering all 4994 Census 2021 London LSOAs
# Step 1: start with census lsoa_code and borough name
census_boroughs = census_london[["lsoa_code", "lsoa_name"]].copy()
census_boroughs["lad_name"] = census_boroughs["lsoa_name"].apply(
    lambda x: next((b for b in london_boroughs if x.startswith(b)), None)
)

# Step 2: merge with existing IMD data using 2021 code
imd_complete = census_boroughs.merge(
    imd_london[["lsoa_code_2021", "imd_rank", "imd_decile"]],
    left_on="lsoa_code",
    right_on="lsoa_code_2021",
    how="left"
).drop(columns=["lsoa_code_2021"])

# Step 3: fill missing rows with borough median
imd_complete = imd_complete.merge(borough_median, on="lad_name", how="left")

imd_complete["imd_rank"] = imd_complete["imd_rank"].fillna(imd_complete["imd_rank_median"])
imd_complete["imd_decile"] = imd_complete["imd_decile"].fillna(imd_complete["imd_decile_median"])

imd_complete = imd_complete.drop(columns=["imd_rank_median", "imd_decile_median"])
imd_complete["imd_rank"] = imd_complete["imd_rank"].astype(int)
imd_complete["imd_decile"] = imd_complete["imd_decile"].astype(int)

# Step 4: add a flag column to mark borough-median-filled rows
imd_complete["imd_imputed"] = imd_complete["lsoa_code"].isin(in_census_not_imd).astype(int)

print("=== IMD Complete ===")
print("Shape:", imd_complete.shape)
print(f"Imputed rows: {imd_complete['imd_imputed'].sum()}")
print(f"Missing values remaining: {imd_complete[['imd_rank','imd_decile']].isna().sum().sum()}")
print()
print(imd_complete.head(3).to_string())

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/imd_london_clean.csv")
imd_complete.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== IMD Complete ===
Shape: (5016, 6)
Imputed rows: 181
Missing values remaining: 0

   lsoa_code            lsoa_name        lad_name  imd_rank  imd_decile  imd_imputed
0  E01000001  City of London 001A  City of London     29199           9            0
1  E01000002  City of London 001B  City of London     30379          10            0
2  E01000003  City of London 001C  City of London     14915           5            0

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/imd_london_clean.csv


##  Fix duplicates by averaging

In [23]:
# Some 2021 LSOAs map to multiple 2011 LSOAs in IMD, causing duplicate rows
# Fix: aggregate by taking the mean imd_rank and imd_decile, round to int
imd_complete = imd_complete.groupby(["lsoa_code", "lsoa_name", "lad_name", "imd_imputed"], as_index=False).agg(
    imd_rank=("imd_rank", "mean"),
    imd_decile=("imd_decile", "mean")
)

imd_complete["imd_rank"] = imd_complete["imd_rank"].round(0).astype(int)
imd_complete["imd_decile"] = imd_complete["imd_decile"].round(0).astype(int)

# Reorder columns
imd_complete = imd_complete[["lsoa_code", "lsoa_name", "lad_name",
                              "imd_rank", "imd_decile", "imd_imputed"]]

print("=== IMD Complete (deduplicated) ===")
print("Shape:", imd_complete.shape)
print(f"Duplicate rows remaining: {imd_complete.duplicated(subset=['lsoa_code']).sum()}")
print(f"Missing values: {imd_complete[['imd_rank','imd_decile']].isna().sum().sum()}")
print()
print(imd_complete.head(3).to_string())

# Overwrite saved file
output_path = os.path.join(BASE, "05_processed/imd_london_clean.csv")
imd_complete.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== IMD Complete (deduplicated) ===
Shape: (4994, 6)
Duplicate rows remaining: 0
Missing values: 0

   lsoa_code            lsoa_name        lad_name  imd_rank  imd_decile  imd_imputed
0  E01000001  City of London 001A  City of London     29199           9            0
1  E01000002  City of London 001B  City of London     30379          10            0
2  E01000003  City of London 001C  City of London     14915           5            0

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/imd_london_clean.csv


## Inspect OpenStreetEV for duplicates and coordinate validity

In [24]:
# Check for duplicate location_id and validate coordinates within Greater London bounding box
# Greater London approximate bounding box:
# Latitude:  51.28 – 51.70
# Longitude: -0.51 – 0.33

print("=== OpenStreetEV Basic Checks ===")
print(f"Total rows: {len(osev)}")
print(f"Duplicate location_id: {osev.duplicated(subset=['location_id']).sum()}")
print(f"Missing latitude: {osev['latitude'].isna().sum()}")
print(f"Missing longitude: {osev['longitude'].isna().sum()}")
print()

# Flag coordinates outside London bounding box
outside_bbox = osev[
    (osev["latitude"]  < 51.28) | (osev["latitude"]  > 51.70) |
    (osev["longitude"] < -0.51) | (osev["longitude"] > 0.33)
]
print(f"Rows outside London bounding box: {len(outside_bbox)}")
print()
print(outside_bbox[["location_id", "latitude", "longitude", "borough"]].head(5).to_string())

=== OpenStreetEV Basic Checks ===
Total rows: 23045
Duplicate location_id: 30
Missing latitude: 0
Missing longitude: 0

Rows outside London bounding box: 0

Empty DataFrame
Columns: [location_id, latitude, longitude, borough]
Index: []


## Remove duplicate location_id rows

In [26]:
# All duplicates are exact row copies — safe to drop duplicates directly
osev_clean = osev.drop_duplicates(subset=["location_id"]).reset_index(drop=True)

print("=== OpenStreetEV Cleaned ===")
print(f"Rows before: {len(osev)}")
print(f"Rows after:  {len(osev_clean)}")
print(f"Duplicates remaining: {osev_clean.duplicated(subset=['location_id']).sum()}")
print(f"Missing values:\n{osev_clean.isnull().sum()}")

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/osev_london_clean.csv")
osev_clean.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== OpenStreetEV Cleaned ===
Rows before: 23045
Rows after:  23015
Duplicates remaining: 0
Missing values:
country_code         0
location_id          0
external_uuid        2
location_name        1
address              7
postcode             0
borough              0
latitude             0
longitude            0
location_category    0
location_type        0
dtype: int64

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/osev_london_clean.csv


## Clean EVSE Status: aggregate 25M rows into per-EVSE availability rate

In [27]:
# Aggregate 25 million status records into per-EVSE availability rate
# Status values: typically 'AVAILABLE', 'CHARGING', 'OUTOFORDER', 'UNKNOWN' etc.
# First check what status values exist

print("=== EVSE Status value counts ===")
print(evse_status["status"].value_counts())
print()
print("Sample rows:")
print(evse_status.head(5).to_string())

=== EVSE Status value counts ===
status
AVAILABLE      12022386
CHARGING        8969069
OUTOFORDER      2047816
UNKNOWN         1904427
BLOCKED          331173
INOPERATIVE      102681
RESERVED          84010
Name: count, dtype: int64

Sample rows:
            uid                          evse_uid         last_updated     status
0  o-1765736769  0002271b5e174a428d01e9847b69d195  2025-09-02 10:49:21  AVAILABLE
1  o-1795225869  0003eb1b04bbe87d2a4bbd95ea85fbde  2025-09-19 07:05:48  AVAILABLE
2  c-1732589905  0007804415dd5bf2c01750f4bf0c4938  2025-08-06 04:42:54  AVAILABLE
3  o-1339343093  001255f66bcb442fa81d8e7fdbec81ab  2025-01-17 12:13:32  AVAILABLE
4  o-1416124652  001255f66bcb442fa81d8e7fdbec81ab  2025-03-04 05:19:06  AVAILABLE


## Aggregate EVSE status into availability rate per EVSE

In [28]:
# Aggregate 25M rows into per-EVSE availability and utilisation rates
# availability_rate = AVAILABLE / total records
# utilisation_rate  = CHARGING / total records

evse_agg = evse_status.groupby("evse_uid")["status"].value_counts().unstack(fill_value=0).reset_index()

# Ensure all expected status columns exist even if absent in data
for col in ["AVAILABLE", "CHARGING", "OUTOFORDER", "UNKNOWN", "BLOCKED", "INOPERATIVE", "RESERVED"]:
    if col not in evse_agg.columns:
        evse_agg[col] = 0

# Calculate total records per EVSE
evse_agg["total_records"] = evse_agg[["AVAILABLE", "CHARGING", "OUTOFORDER",
                                       "UNKNOWN", "BLOCKED", "INOPERATIVE", "RESERVED"]].sum(axis=1)

# Calculate rates
evse_agg["availability_rate"] = (evse_agg["AVAILABLE"] / evse_agg["total_records"]).round(4)
evse_agg["utilisation_rate"]  = (evse_agg["CHARGING"]  / evse_agg["total_records"]).round(4)

# Keep only useful columns
evse_clean = evse_agg[["evse_uid", "total_records", "availability_rate", "utilisation_rate"]].copy()

print("=== EVSE Status Aggregated ===")
print("Shape:", evse_clean.shape)
print()
print(evse_clean.head(5).to_string())
print()
print("availability_rate range:", evse_clean["availability_rate"].min(), "–", evse_clean["availability_rate"].max())
print("utilisation_rate range: ", evse_clean["utilisation_rate"].min(),  "–", evse_clean["utilisation_rate"].max())

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/evse_availability_clean.csv")
evse_clean.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== EVSE Status Aggregated ===
Shape: (28865, 4)

status                          evse_uid  total_records  availability_rate  utilisation_rate
0       0001f0b862005b9b56f409ea583cd4fc             67             0.5075            0.4478
1       0002271b5e174a428d01e9847b69d195            376             0.4947            0.2154
2       0002729a6be20a87309637692e5a3af3             36             0.5000            0.1667
3       0003081cf555fade0d74f080d8baa089            186             0.4839            0.2796
4       0003eb1b04bbe87d2a4bbd95ea85fbde             33             0.4848            0.1212

availability_rate range: 0.0 – 1.0
utilisation_rate range:  0.0 – 1.0

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/evse_availability_clean.csv


## Clean Device UK: filter to GLA area only

In [29]:
# Filter device_all_uk to GLA area only using location_ids from osev_clean
gla_location_ids = set(osev_clean["location_id"])

device_gla = device[device["location_id"].isin(gla_location_ids)].reset_index(drop=True)

print("=== Device UK filtered to GLA ===")
print(f"Rows before: {len(device)}")
print(f"Rows after:  {len(device_gla)}")
print()
print("power_band value counts:")
print(device_gla["power_band"].value_counts())
print()
print(device_gla.head(5).to_string())

# Save to 05_processed/
output_path = os.path.join(BASE, "05_processed/device_gla_clean.csv")
device_gla.to_csv(output_path, index=False)
print()
print("Saved to:", output_path)

=== Device UK filtered to GLA ===
Rows before: 133263
Rows after:  33249

power_band value counts:
power_band
1. Slow           25893
2. Fast (AC)       4792
3. Rapid           1198
4. Ultra-rapid     1141
2. Fast (DC)        225
Name: count, dtype: int64

  location_id zapmap_device_uid power_band  device_max_power
0     6SO91US           98ANCHF    1. Slow              2760
1     LOPDTFA           QYY5JSI    1. Slow              2760
2     UTC0HJS           2VEZNZJ    1. Slow              2760
3     B4QJ4NO           DLP3M40    1. Slow              2990
4     Y9RMDL3           YEU1W4B    1. Slow              2990

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/device_gla_clean.csv


##  Final summary of all cleaned files

In [30]:
# Final summary of all cleaned datasets saved to 05_processed/
import os

processed_dir = os.path.join(BASE, "05_processed")
files = [f for f in os.listdir(processed_dir) if f.endswith(".csv")]

print("=== Cleaned Files in 05_processed/ ===")
print()
for f in sorted(files):
    path = os.path.join(processed_dir, f)
    df = pd.read_csv(path)
    print(f"{f}")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Columns: {df.columns.tolist()}")
    print()

=== Cleaned Files in 05_processed/ ===

census_london_clean.csv
  Shape: 4,994 rows × 10 cols
  Columns: ['lsoa_code', 'lsoa_name', 'total_residents', 'cars_0', 'cars_1', 'cars_2', 'cars_3plus', 'total_households', 'total_cars', 'car_ownership_rate']

device_gla_clean.csv
  Shape: 33,249 rows × 4 cols
  Columns: ['location_id', 'zapmap_device_uid', 'power_band', 'device_max_power']

evse_availability_clean.csv
  Shape: 28,865 rows × 4 cols
  Columns: ['evse_uid', 'total_records', 'availability_rate', 'utilisation_rate']

imd_london_clean.csv
  Shape: 4,994 rows × 6 cols
  Columns: ['lsoa_code', 'lsoa_name', 'lad_name', 'imd_rank', 'imd_decile', 'imd_imputed']

osev_london_clean.csv
  Shape: 23,015 rows × 11 cols
  Columns: ['country_code', 'location_id', 'external_uuid', 'location_name', 'address', 'postcode', 'borough', 'latitude', 'longitude', 'location_category', 'location_type']

